# FerroSim — Grammar — Version 3

Anaconda-compatible, offline, standalone Jupyter notebook with common figure protocol **F01–F20**. Requires a fresh Python 3.11 Jupyter kernel with numpy, pandas, scipy, matplotlib, numba, torch, IPython. Under Anaconda Navigator, open JupyterLab and choose **Run All**. All embedded code is emitted into a dedicated runtime folder; no Github download occurs at runtime.

**Original algorithm preserved:** variable-length physical operation grammar with simple/full subgrammar × GA/QD methods. All scientific rewards come from the original FerroSim RK4 physical oracle. The GFlowNet/diffusion-inspired Segments variants are *heuristics*, not trained GFlowNet or diffusion models.

**Configuration:** N_DISORDERS includes the historical reference disorder plus N−1 random spatial rearrangements; any positive N supported. N_SEEDS and GENERATIONS independent. QUICK has a short search while FULL has the same scientific acquisition budget independent of N. Physical curl has a boundary mismatch, and true novelty or global optimality is *not* certified.

**Figure architecture:** F01 disorder geometry; F02 evaluated reward history; F03 reference Pareto; F04 per-disorder champion waveform evolution; F05 best/simple controls; F06 field/complexity history; F07 predictive calibration; F08 physical polarization maps; F09 dynamic curl and polarization; F10 fidelity and sampling; F11–F18 ensemble only; F19 waveform history + Pareto for each disorder; F20 grid of Pareto fronts across disorder realizations.

In [ ]:
import os,sys,json,hashlib,subprocess,base64,zlib
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
# USER CONTROLS — change these values, restart kernel, then Run All.
MODE="QUICK"              # QUICK or FULL
N_DISORDERS=1             # any positive integer; 1 or 20 or arbitrary N
N_SEEDS=1                 # independent optimization seeds
GENERATIONS=1            # default short test; 32 for publication study
INITIAL_CONTROLS=8 if MODE=="QUICK" else 64
BATCH_SIZE=4 if MODE=="QUICK" else 32
CANDIDATES=16 if MODE=="QUICK" else 256
RUN_TRANSFER=False       # expensive N*N champion reevaluations
RUN_REFINEMENT=False     # expensive higher-fidelity check
RUN_PHYSICAL_DIAGNOSTICS=True
PLOT_ALL_DISORDERS=True
assert MODE in ("QUICK","FULL") and N_DISORDERS>=1 and N_SEEDS>=1
assert GENERATIONS>=0 and CANDIDATES>=BATCH_SIZE and INITIAL_CONTROLS>=1
ROOT=Path.cwd()/"FerroSim_Grammar_v3";ROOT.mkdir(parents=True,exist_ok=True)
print("Anaconda/Jupyter runtime:",ROOT.resolve())

In [ ]:
%%writefile FerroSim_Grammar_v3/representation.py
"""Variable-length procedural control grammar; each primitive has an explicit physical meaning.
Tokens are (operation ID, relative duration, target, amplitude, frequency/cycles, decay time).
A fixed-width encoding is storage only: inactive tokens are ignored by the compiler.
"""
import numpy as np
from scipy.signal import find_peaks
OPS=('ramp','hold','reverse','pulse','ringdown','burst')
RAMP,HOLD,REVERSE,PULSE,RINGDOWN,BURST=range(6)
MIN_K,MAX_K=2,12
STRIDE=6
DIM=1+MAX_K*STRIDE
DRIVE_SAMPLES=900

def encode(ops):
    k=len(ops);assert MIN_K<=k<=MAX_K
    z=np.zeros(DIM);z[0]=k
    z[1:1+STRIDE*k]=np.asarray(ops,float).reshape(-1)
    return z

def decode(z):
    z=np.asarray(z,float).ravel();k=int(np.clip(round(z[0]),MIN_K,MAX_K))
    ops=z[1:1+STRIDE*k].reshape(k,STRIDE).copy()
    ops[:,0]=np.clip(np.rint(ops[:,0]),0,5)
    ops[:,1]=np.clip(ops[:,1],.03,5)
    ops[:,2:4]=np.clip(ops[:,2:4],-70,70)
    ops[:,4]=np.clip(ops[:,4],.1,12)
    ops[:,5]=np.clip(ops[:,5],.08,5)
    return ops

def compile_one(z,n=900,return_meta=False):
    seq=decode(z);u=np.linspace(0,1,n);wave=np.zeros(n)
    d=np.clip(seq[:,1],.03,5);d=d/d.sum()*.9
    bounds=.05+np.r_[0,np.cumsum(d)];bounds[-1]=.95
    level=0.
    for i,(op,_,target,amp,freq,tau) in enumerate(seq):
        left,right=bounds[i:i+2]
        mask=(u>=left)&(u<=right if i==len(seq)-1 else u<right)
        q=np.clip((u[mask]-left)/(right-left),0,1)
        op=int(op)
        if op==RAMP:
            f=level+(target-level)*q;newlevel=target
        elif op==REVERSE:
            newlevel=-np.sign(level if abs(level)>.01 else (target if abs(target)>.01 else 1))*abs(target)
            f=level+(newlevel-level)*q
        elif op==HOLD:
            f=np.full(q.shape,level);newlevel=level
        elif op==PULSE:
            f=level+amp*np.sin(np.pi*q)**2;newlevel=level
        elif op==RINGDOWN:
            # Oscillatory ringdown gated smoothly to ensure continuity at both ends.
            envelope=np.sin(np.pi*q)**2*np.exp(-q/tau)
            f=level+amp*envelope*np.sin(2*np.pi*freq*q);newlevel=level
        else:
            f=level+amp*np.sin(np.pi*q)**2*np.sin(2*np.pi*freq*q);newlevel=level
        wave[mask]=f;level=float(newlevel)
    # Closing ramp: fixed 0.025 duration, then zero field and slope at end.
    close=(u>.95)&(u<.975);wave[close]=level*(1-(u[close]-.95)/.025)
    if return_meta:return u,wave,bounds
    return wave

def fields(x,n=900):
    z=np.atleast_2d(np.asarray(x,float));return np.stack([compile_one(y,n) for y in z])

def complexity(x,n=1000):
    z=np.atleast_2d(x);w=fields(z,n)
    result={k:np.zeros(len(z),dtype=int) for k in ['segments','effective_segments','turns','raw_turns','turns_1pct','turns_5pct','n_ramps','n_pulses','n_ringdowns','n_bursts','n_holds','n_reverses','parameter_count']}
    for i,x in enumerate(z):
        ops=decode(x);k=len(ops);result['segments'][i]=k;result['effective_segments'][i]=k
        for label,idx in [('n_ramps',RAMP),('n_pulses',PULSE),('n_ringdowns',RINGDOWN),('n_bursts',BURST),('n_holds',HOLD),('n_reverses',REVERSE)]:result[label][i]=np.sum(ops[:,0]==idx)
        result['parameter_count'][i]=2*k+sum(int(o in [PULSE,RINGDOWN,BURST])*(1+int(o in [RINGDOWN,BURST])*2) for o in ops[:,0])
        peak=np.ptp(w[i]);
        for key,prom in [('raw_turns',0),('turns_1pct',.01),('turns',.02),('turns_5pct',.05)]:
            result[key][i]=int(len(find_peaks(w[i],prominence=peak*prom)[0])+len(find_peaks(-w[i],prominence=peak*prom)[0])) if peak else 0
    result['peak_ec']=np.max(np.abs(w),axis=1);result['rms_ec']=np.sqrt(np.mean(w*w,axis=1))
    return result

def random_op(rng,grammar='full'):
    available=[RAMP,HOLD,REVERSE] if grammar=='simple' else list(range(6))
    op=int(rng.choice(available));return np.array([op,float(rng.uniform(.25,1.6)),float(rng.uniform(-40,40)),float(rng.uniform(-35,35)),float(rng.uniform(.5,7)),float(rng.uniform(.15,1.7))])

def sample_one(rng,grammar='full',k=None):
    k=int(rng.integers(MIN_K,MAX_K+1)) if k is None else int(k)
    ops=[random_op(rng,grammar) for _ in range(k)];ops[0][0]=RAMP
    return encode(ops)

def initial_design(seed,n=8,grammar='full'):
    rng=np.random.default_rng(seed);ans=[]
    # shared initial candidates across both grammars, for defensible ablation
    for peak in (10.,-10.,25.,-25.,40.,-40.):
        ops=np.array([[RAMP,.9,peak,0,2,1],[HOLD,.3,0,0,2,1],[RAMP,.9,0,0,2,1]],float)
        ans.append(encode(ops))
        if len(ans)>=n:return np.array(ans)
    while len(ans)<n:ans.append(sample_one(rng,grammar))
    return np.asarray(ans)

def mutate(z,rng,grammar='full',force=None):
    ops=decode(z);k=len(ops)
    choices=['insert','delete','replace','duration','target','amplitude','frequency','decay','swap','rescale']
    op=force if force else str(rng.choice(choices,p=[.13,.13,.14,.14,.14,.12,.06,.05,.04,.05]))
    j=int(rng.integers(k))
    if op=='insert' and k<MAX_K:
        ops=np.insert(ops,j,random_op(rng,grammar),axis=0)
    elif op=='delete' and k>MIN_K:
        ops=np.delete(ops,j,axis=0)
    elif op=='replace':ops[j]=random_op(rng,grammar)
    elif op=='duration':ops[j,1]*=np.exp(rng.normal(0,.45))
    elif op=='target':ops[j,2]+=rng.normal(0,10)
    elif op=='amplitude':ops[j,3]+=rng.normal(0,10)
    elif op=='frequency':ops[j,4]*=np.exp(rng.normal(0,.3))
    elif op=='decay':ops[j,5]*=np.exp(rng.normal(0,.4))
    elif op=='swap' and k>2:ops[[j,(j+1)%k]]=ops[[(j+1)%k,j]]
    elif op=='rescale':ops[:,2:4]*=np.exp(rng.normal(0,.3))
    if grammar=='simple':
        for row in ops:
            if int(row[0])>REVERSE:row[0]=int(rng.choice([RAMP,HOLD,REVERSE]))
    ops[:,1]=np.clip(ops[:,1],.03,5)
    ops[:,2:4]=np.clip(ops[:,2:4],-70,70)
    ops[:,4]=np.clip(ops[:,4],.1,12);ops[:,5]=np.clip(ops[:,5],.08,5)
    return encode(ops),op


In [ ]:
%%writefile FerroSim_Grammar_v3/oracle.py
"""Accelerated deterministic FerroSim GLD evaluator.

Physics: native rama_dev_updated@5eb258e, gamma=1, tetragonal, k=100,
dep_alpha=.3, finite-grid interactions with fixed /4 neighbor normalization.
Default integration is subdivided RK4, NOT the native first-order Euler method.
No field-amplitude bound or state clipping is imposed in RK4. The native
calDeriv clipping guard |P|=10 is explicitly diagnosed if ever crossed.
Native periodic central-difference curl and an interior diagnostic are returned.
"""
from __future__ import annotations
import os
import math
from pathlib import Path
import numpy as np
os.environ.setdefault("NUMBA_NUM_THREADS", "2")
from numba import njit, prange

N = 20
ALPHA1, ALPHA2, ALPHA3 = -1.6, 12.2, 40.0
COUPLING, DEP_ALPHA = 100.0, 0.3
EC = (-2.0 / 3.0) * ALPHA1 * np.sqrt(abs(-ALPHA1/(3.0*ALPHA2)))
TIME = np.linspace(0.0, 4.0, 1200)[:950]
DT = float(TIME[1]-TIME[0])
SOURCE_COMMIT = "5eb258e2a47d10b3d9bdb9e94dd9879b8410024c"
_LOCAL_DEFECT_PATH = Path(os.environ.get("FERROSIM_DEFECT_PATH", str(Path(__file__).with_name("defect_list_seed.npy"))))
DEFAULT_DEFECT_PATH = (_LOCAL_DEFECT_PATH if _LOCAL_DEFECT_PATH.exists()
                       else Path(__file__).resolve().parents[1]/"GA-DKL/defect_list_seed.npy")
_DEFECTS = None


def default_defects():
    global _DEFECTS
    if _DEFECTS is None:
        _DEFECTS = np.load(DEFAULT_DEFECT_PATH, allow_pickle=False).astype(np.float64).reshape(N,N,2)
    return _DEFECTS.copy()


@njit(cache=True)
def _rhs(p, ex, defects, coupling, dep_alpha, native_clip):
    n = p.shape[1]
    out = np.empty_like(p)
    mx, my = 0.0, 0.0
    for i in range(n):
        for j in range(n):
            mx += p[0,i,j]
            my += p[1,i,j]
    mx /= n*n
    my /= n*n
    for i in range(n):
        for j in range(n):
            x, y = p[0,i,j], p[1,i,j]
            if native_clip:
                x = min(10.,max(-10.,x))
                y = min(10.,max(-10.,y))
            sx, sy = 0.0, 0.0
            # Same finite-grid graph as native knn_search(r=1.1).
            if i>0:
                sx += p[0,i-1,j]; sy += p[1,i-1,j]
            if j>0:
                sx += p[0,i,j-1]; sy += p[1,i,j-1]
            if j<n-1:
                sx += p[0,i,j+1]; sy += p[1,i,j+1]
            if i<n-1:
                sx += p[0,i+1,j]; sy += p[1,i+1,j]
            out[0,i,j] = -(2*ALPHA1*x + 4*ALPHA2*x*x*x + 2*ALPHA3*y*y*x
                           + coupling*(x-sx/4.) - ex + dep_alpha*mx - defects[i,j,0])
            out[1,i,j] = -(2*ALPHA1*y + 4*ALPHA2*y*y*y + 2*ALPHA3*x*x*y
                           + coupling*(y-sy/4.) + dep_alpha*my - defects[i,j,1])
    return out


@njit(cache=True)
def _maxabs(p):
    q = 0.0
    for k in range(p.size):
        v = abs(p.flat[k])
        if not np.isfinite(v):
            return np.inf
        if v>q:
            q=v
    return q


@njit(cache=True)
def _curl(p):
    n=p.shape[1]
    c=np.empty((n,n),dtype=np.float64)
    total, interior = 0.,0.
    for i in range(n):
        for j in range(n):
            v=.5*(p[1,(i+1)%n,j]-p[1,(i-1)%n,j]-p[0,i,(j+1)%n]+p[0,i,(j-1)%n])
            c[i,j]=v
            total+=abs(v)
            if 0<i<n-1 and 0<j<n-1:
                interior+=abs(v)
    return total,interior,c


@njit(cache=True)
def _integrate(field, defects, initial, fidelity, method, history, coupling, dep_alpha, max_substeps):
    """method 0: native-style Euler, method 1: curvature-subdivided RK4."""
    p=initial.copy()
    steps=field.size
    n=p.shape[1]
    hst=np.full((2,steps,n,n),np.nan,dtype=np.float64) if history else np.empty((2,1,n,n),dtype=np.float64)
    hst[:,0,:,:]=p
    peak=_maxabs(p)
    defects_peak=0.
    for q in range(defects.size):
        defects_peak=max(defects_peak,abs(defects.flat[q]))
    rhs_calls=0
    subdivisions=0
    guard_exceeded=peak>10.
    status=0
    for t in range(1,steps):
        e0,e1=field[t-1]*EC,field[t]*EC
        if method==0:
            count=fidelity
        else:
            # A conservative estimate of the local Jacobian row-sum magnitude.
            # The forcing equilibrium scale anticipates sharp field increases.
            pbound=max(_maxabs(p),((max(abs(e0),abs(e1))+defects_peak+10.)/(4*ALPHA2))**(1/3)+.3)
            curvature=(12*ALPHA2+6*abs(ALPHA3))*pbound*pbound+2*coupling+abs(2*ALPHA1)+dep_alpha
            needed=DT*curvature*fidelity
            # Check in floating point before integer conversion: finite but huge
            # requested fields can otherwise overflow the subdivision counter.
            if not np.isfinite(needed) or needed>max_substeps:
                status=2
                break
            count=max(2*fidelity,int(math.ceil(needed)))
        if count>max_substeps:
            status=2
            break
        h=DT/count
        for sub in range(count):
            f0=sub/count
            f1=(sub+1)/count
            if method==0:
                # At fidelity=1 this exactly uses field[t], as native calcODE does.
                ex=e0+(e1-e0)*f1
                p=p+h*_rhs(p,ex,defects,coupling,dep_alpha,True)
                rhs_calls+=1
            else:
                ea=e0+(e1-e0)*f0
                eb=e0+(e1-e0)*f1
                em=.5*(ea+eb)
                k1=_rhs(p,ea,defects,coupling,dep_alpha,False)
                k2=_rhs(p+.5*h*k1,em,defects,coupling,dep_alpha,False)
                k3=_rhs(p+.5*h*k2,em,defects,coupling,dep_alpha,False)
                k4=_rhs(p+h*k3,eb,defects,coupling,dep_alpha,False)
                p=p+(h/6.)*(k1+2*k2+2*k3+k4)
                rhs_calls+=4
            pm=_maxabs(p)
            if not np.isfinite(pm):
                status=1
                break
            peak=max(peak,pm)
            guard_exceeded=guard_exceeded or pm>10.
        subdivisions+=count
        if history:
            hst[:,t,:,:]=p
        if status:
            break
    if not history:
        hst[:,0,:,:]=p
    if status:
        return np.nan,np.nan,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst
    reward,interior,_=_curl(p)
    return reward,interior,p,peak,guard_exceeded,status,rhs_calls,subdivisions,hst


@njit(cache=True,parallel=True)
def _batch(fields,defects,initial,fidelity,method,coupling,dep_alpha,max_substeps):
    batch=fields.shape[0]
    n=initial.shape[1]
    reward=np.empty(batch); interior=np.empty(batch); peak=np.empty(batch)
    final=np.empty((batch,2,n,n)); guards=np.empty(batch,dtype=np.bool_)
    status=np.empty(batch,dtype=np.int64); rhs=np.empty(batch,dtype=np.int64); subdivisions=np.empty(batch,dtype=np.int64)
    for q in prange(batch):
        r,j,p,m,g,s,nrhs,nsub,_=_integrate(fields[q],defects,initial,fidelity,method,False,coupling,dep_alpha,max_substeps)
        reward[q]=r; interior[q]=j; final[q]=p; peak[q]=m
        guards[q]=g; status[q]=s; rhs[q]=nrhs; subdivisions[q]=nsub
    return reward,interior,final,peak,guards,status,rhs,subdivisions


def evaluate_fields(fields_ec, fidelity=1, return_history=False, method="rk4", defects_ec=None,
                    initial_p=None, coupling=COUPLING, dep_alpha=DEP_ALPHA, max_substeps=100000):
    """Evaluate x-only waveforms with a fresh state for every candidate.

    Parameters
    ----------
    fields_ec : (batch,900) or (batch,950), or a single corresponding vector.
        Actual native-interface field E/Ec, WITHOUT normalization or amplitude cap.
        With 900 samples, 50 repeats of the final field are appended.
    fidelity : integer >=1
        Multiplies RK4 temporal resolution. Legacy Euler uses this many substeps.
    return_history : bool
        Add P_history shape (batch,2,950,n,n), mean_P and curl_history.
    method : "rk4" (default) or "legacy_euler"
        RK4 uses linear field interpolation and no state clipping.
    initial_p : optional (n,n,2) or (2,n,n), default zeros on20x20.
    defects_ec : optional (n*n,2) or (n,n,2), default published19-site array.

    status=0 success; 1 nonfinite integration; 2 subdivision resource limit.
    native_guard_exceeded flags |P|>10 without clipping or discarding the result.
    """
    arr=np.asarray(fields_ec,dtype=np.float64)
    if arr.ndim==1:
        arr=arr[None,:]
    if arr.ndim!=2 or arr.shape[1] not in (900,950):
        raise ValueError("Expected (batch,900) or (batch,950) native E/Ec values")
    if not np.isfinite(arr).all():
        raise ValueError("All supplied fields must be finite")
    if arr.shape[1]==900:
        arr=np.concatenate((arr,np.repeat(arr[:,-1:],50,axis=1)),axis=1)
    arr=np.ascontiguousarray(arr)
    if int(fidelity)!=fidelity or fidelity<1:
        raise ValueError("fidelity must be an integer >=1")
    if int(max_substeps)!=max_substeps or max_substeps<1:
        raise ValueError("max_substeps must be an integer >=1")
    if method not in ("rk4","legacy_euler"):
        raise ValueError("Unknown integration method")
    method_id=int(method=="rk4")
    if initial_p is None:
        initial=np.zeros((2,N,N),dtype=np.float64)
    else:
        initial=np.asarray(initial_p,dtype=np.float64)
        if initial.shape[-1]==2 and initial.shape[0]!=2:
            initial=initial.transpose(2,0,1)
        if initial.ndim!=3 or initial.shape[0]!=2 or initial.shape[1]!=initial.shape[2]:
            raise ValueError("Invalid initial polarization shape")
        initial=np.ascontiguousarray(initial)
    n=initial.shape[1]
    defects=default_defects() if defects_ec is None else np.asarray(defects_ec,dtype=np.float64).reshape(n,n,2)
    if defects.shape!=(n,n,2):
        raise ValueError("Defect grid must match initial polarization grid")
    if not np.isfinite(initial).all() or not np.isfinite(defects).all():
        raise ValueError("Initial polarization and defects must be finite")
    defects=np.ascontiguousarray(defects*EC)
    if return_history:
        out=[_integrate(x,defects,initial,int(fidelity),method_id,True,float(coupling),float(dep_alpha),int(max_substeps)) for x in arr]
        r,j,p,m,g,s,nrhs,nsub,hist=map(np.asarray,zip(*out))
    else:
        r,j,p,m,g,s,nrhs,nsub=_batch(arr,defects,initial,int(fidelity),method_id,float(coupling),float(dep_alpha),int(max_substeps))
    result={"reward":r,"curl":r,"interior":j,"final_P":p,"max_abs_P":m,
            "native_guard_exceeded":g,"status":s,"valid":(s==0)&np.isfinite(r),
            "rhs_calls":nrhs,"substeps":nsub,"field_peak_ec":np.max(np.abs(arr),axis=1),
            "fidelity":int(fidelity),"method":method}
    if return_history:
        result["P_history"]=hist
        result["mean_P"]=hist.mean(axis=(3,4))
        ch=np.empty((arr.shape[0],arr.shape[1]))
        ih=np.empty_like(ch)
        for b in range(arr.shape[0]):
            for t in range(arr.shape[1]):
                ch[b,t],ih[b,t],_=_curl(hist[b,:,t])
        result["curl_history"]=ch
        result["interior_history"]=ih
        result["time"]=TIME.copy()
        result["field_ec"]=arr
    return result


def curl_field(p):
    """Native periodic centered curl diagnostic for P shaped (2,n,n)."""
    return _curl(np.asarray(p,dtype=np.float64))[2]


In [ ]:
%%writefile FerroSim_Grammar_v3/dkl.py
"""Nonlinear neural feature kernel + exact GP trained on fixed waveform samples."""
import numpy as np
import torch
from torch import nn
class DKL(nn.Module):
    def __init__(self,seed=0):
        super().__init__();torch.manual_seed(seed)
        self.net=nn.Sequential(nn.Linear(65,32),nn.Tanh(),nn.Linear(32,8)).double()
        self.logl=nn.Parameter(torch.tensor(0.,dtype=torch.double));self.logsig=nn.Parameter(torch.tensor(0.,dtype=torch.double))
        self.lognoise=nn.Parameter(torch.tensor(-2.,dtype=torch.double));self.optim=torch.optim.Adam(self.parameters(),lr=.015)
        self.cache=None
    def features(self,w):
        # 64 sampled waveform values plus segment count
        return self.net(torch.asinh(w))
    def kernel(self,a,b):
        dist=torch.cdist(a,b).square().clamp_min(0);return torch.exp(2*self.logsig)*torch.exp(-dist/(2*torch.exp(2*self.logl)))
    def fit(self,X,y,steps=35):
        from representation import fields,complexity
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        z=np.c_[w/30.,k/16.];self.ycenter=float(np.mean(y));self.yscale=max(float(np.std(y)),1e-3)
        self.X=torch.tensor(z,dtype=torch.double);self.y=torch.tensor((np.asarray(y)-self.ycenter)/self.yscale,dtype=torch.double)
        self.train()
        for _ in range(steps):
            self.optim.zero_grad();f=self.features(self.X);K=self.kernel(f,f);noise=torch.exp(2*self.lognoise)+1e-5
            L=torch.linalg.cholesky(K+noise*torch.eye(len(X),dtype=torch.double))
            alpha=torch.cholesky_solve(self.y[:,None],L)
            loss=(.5*(self.y[:,None]*alpha).sum()+torch.log(torch.diag(L)).sum())/len(X)
            loss.backward();self.optim.step()
            with torch.no_grad():self.logl.clamp_(-4,4);self.logsig.clamp_(-4,4);self.lognoise.clamp_(-6,1)
        self.eval()
        with torch.no_grad():
            self.f=self.features(self.X);K=self.kernel(self.f,self.f);self.L=torch.linalg.cholesky(K+(torch.exp(2*self.lognoise)+1e-5)*torch.eye(len(X)))
            self.alpha=torch.cholesky_solve(self.y[:,None],self.L)
        return self
    def predict(self,X):
        from representation import fields,complexity
        if not len(X):return np.array([]),np.array([])
        w=fields(X,64);k=complexity(X)['segments'].reshape(-1,1)
        with torch.no_grad():
            x=torch.tensor(np.c_[w/30.,k/16.],dtype=torch.double);f=self.features(x);c=self.kernel(f,self.f)
            mean=(c@self.alpha).squeeze();sol=torch.linalg.solve_triangular(self.L,c.T,upper=False)
            std=(torch.exp(2*self.logsig)-sol.square().sum(0)).clamp_min(1e-9).sqrt()
            return np.atleast_1d(mean.numpy()*self.yscale+self.ycenter),np.atleast_1d(std.numpy()*self.yscale)


In [ ]:
%%writefile FerroSim_Grammar_v3/experiment.py
"""Grammar-based FerroSim optimization; all rows correspond to actual physical simulator calls."""
import os,time,argparse,json
os.environ.setdefault('NUMBA_NUM_THREADS','1');os.environ.setdefault('OMP_NUM_THREADS','1');os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
from pathlib import Path
import numpy as np,pandas as pd
from representation import OPS,decode,fields,initial_design,sample_one,mutate,complexity
from oracle import evaluate_fields,default_defects
from dkl import DKL
METHODS=['GA-DKL','QD-DKL']
GRAMMARS=['simple','full']

def disorder_maps(folder,n=20):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    ref=default_defects();active=ref.reshape(-1,2);active=active[np.any(active!=0,axis=1)]
    assert len(active)==19
    out={'reference':ref};np.save(folder/'reference.npy',ref)
    for k in range(n-1):
        rng=np.random.default_rng(20261122+k);pos=rng.choice(400,19,replace=False);m=np.zeros((400,2));m[pos]=active[rng.permutation(19)]
        out[f'd{k:02d}']=m.reshape(20,20,2);np.save(folder/f'd{k:02d}.npy',out[f'd{k:02d}'])
    return out

def propose(method,grammar,X,y,rng,count):
    ix=np.argsort(y)[::-1]
    pool=X[ix[:min(32,len(ix))]]
    if method.startswith('QD'):
        comp=complexity(X);labels=[(int(comp['segments'][i]),int(comp['n_ringdowns'][i]+comp['n_bursts'][i])) for i in range(len(X))]
        archive={}
        for i,key in enumerate(labels):
            if key not in archive or y[i]>y[archive[key]]:archive[key]=i
        pool=X[list(archive.values())]
    z=[];ops=[]
    for _ in range(count):
        if rng.random()<.10:
            child=sample_one(rng,grammar);label='restart'
        else:
            parent=pool[int(rng.integers(len(pool)))];child,label=mutate(parent,rng,grammar)
            if rng.random()<.20:child,_=mutate(child,rng,grammar)
        z.append(child);ops.append(label)
    return np.array(z),ops

def eval_controls(z,d,oracle_method='rk4',fidelity=1):
    v=evaluate_fields(fields(z,900),defects_ec=d,method=oracle_method,fidelity=fidelity,max_substeps=100000)
    reward=np.asarray(v['reward']);valid=np.asarray(v['valid']) & ~np.asarray(v['native_guard_exceeded']) & np.isfinite(reward)
    return np.where(valid,reward,np.nan),valid,np.asarray(v['interior']),np.asarray(v['rhs_calls'])

def one_run(mp,defects,method,grammar,seed,out,initial=64,generations=32,batch=32,candidates=256,steps=35,oracle_method='rk4'):
    folder=out/mp/(grammar+'__'+method)/f'seed_{seed}';folder.mkdir(parents=True,exist_ok=True)
    history=folder/'history.csv';serialized=folder/'programs.npz';expected=initial+generations*batch
    if history.exists() and serialized.exists():
        prior=pd.read_csv(history)
        if len(prior)==expected:
            return prior
    rng=np.random.default_rng(9011+seed*10007+int(mp[1:]) *101 if mp.startswith('d') else 9011+seed*10007)
    X=initial_design(seed,initial,grammar);Y=[];rows=[]
    for gen in range(generations+1):
        if gen==0:
            candidates_selected=X;ops=['initial']*len(X);mean=np.full(len(X),np.nan);std=mean.copy()
        else:
            yy=np.asarray(Y);good=np.isfinite(yy)
            pool,opnames=propose(method,grammar,X[good],yy[good],rng,candidates)
            if good.sum()>=2:
                model=DKL(seed+gen);model.fit(X[good],yy[good],steps=steps)
                mu,sig=model.predict(pool)
            else:mu=np.zeros(len(pool));sig=np.ones(len(pool))
            choice=np.argsort(-(mu+2*sig))[:batch]
            candidates_selected=pool[choice];ops=[opnames[int(i)] for i in choice];mean=mu[choice];std=sig[choice]
            X=np.vstack([X,candidates_selected])
        reward,valid,interior,rhs=eval_controls(candidates_selected,defects,oracle_method)
        Y.extend(reward.tolist());comp=complexity(candidates_selected)
        for j,z in enumerate(candidates_selected):
            seq=decode(z);rows.append(dict(map=mp,grammar=grammar,method=method,seed=seed,evaluation=len(rows)+1,generation=gen,reward=float(reward[j]),valid=bool(valid[j]),interior=float(interior[j]),rhs_calls=int(rhs[j]),operations=int(comp['segments'][j]),effective_operations=int(comp['effective_segments'][j]),turns=int(comp['turns'][j]),peak_ec=float(comp['peak_ec'][j]),rms_ec=float(comp['rms_ec'][j]),param_count=int(comp['parameter_count'][j]),n_ringdowns=int(comp['n_ringdowns'][j]),n_bursts=int(comp['n_bursts'][j]),n_pulses=int(comp['n_pulses'][j]),operator=ops[j],predicted_mean=float(mean[j]),predicted_std=float(std[j]),program=' > '.join(OPS[int(o)] for o in seq[:,0])))
        pd.DataFrame(rows).to_csv(history,index=False);np.savez_compressed(serialized,programs=X)
    return pd.DataFrame(rows)

def run(out,maps,seeds=range(3),initial=64,generations=32,batch=32,candidates=256,steps=35,oracle_method='rk4'):
    out=Path(out);out.mkdir(parents=True,exist_ok=True);all=[];start=time.time()
    for name,d in maps.items():
        for seed in seeds:
            for grammar in GRAMMARS:
                for method in METHODS:
                    print(f'{time.time()-start:.1f}s map={name} seed={seed} grammar={grammar} method={method}',flush=True)
                    all.append(one_run(name,d,method,grammar,seed,out,initial,generations,batch,candidates,steps,oracle_method))
    data=pd.concat(all,ignore_index=True);data.to_csv(out/'all_evaluations.csv',index=False)
    print('total measured:',len(data),'valid:',int(data.valid.sum()),'elapsed:',round(time.time()-start,1),flush=True)
    return data

if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--out',default='Grammar_results');ap.add_argument('--pilot',action='store_true');ap.add_argument('--maps',type=int,default=20);ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--initial',type=int,default=64);ap.add_argument('--generations',type=int,default=32);ap.add_argument('--batch',type=int,default=32);ap.add_argument('--candidates',type=int,default=256);ap.add_argument('--steps',type=int,default=35);ap.add_argument('--oracle-method',choices=['rk4','legacy_euler'],default='rk4');args=ap.parse_args()
    if args.pilot:args.maps=1;args.seeds=1;args.initial=8;args.generations=1;args.batch=4;args.candidates=16;args.steps=6;args.oracle_method='legacy_euler'
    out=Path(args.out);maps=disorder_maps(out/'disorder_maps',args.maps)
    run(out,maps,range(args.seeds),args.initial,args.generations,args.batch,args.candidates,args.steps,args.oracle_method)


In [ ]:
%%writefile FerroSim_Grammar_v3/analysis.py
"""Analyzes strictly measured grammar-optimizer evaluations, never synthetic performance data."""
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from representation import decode,fields,OPS,complexity
from oracle import evaluate_fields

def load(folder):
    f=Path(folder)/'all_evaluations.csv'
    if not f.exists():raise FileNotFoundError(f'Run experiment first: {f}')
    return pd.read_csv(f)

def get_program(folder,row):
    f=Path(folder)/str(row['map'])/(str(row['grammar'])+'__'+str(row['method']))/f"seed_{int(row['seed'])}"/'programs.npz'
    with np.load(f) as data:return data['programs'][int(row['evaluation'])-1].copy()

def summary(df):
    out=[]
    for key,g in df.groupby(['map','grammar','method','seed']):
        g=g[g.valid & np.isfinite(g.reward)]
        if len(g):
            b=g.loc[g.reward.idxmax()];out.append(dict(zip(['map','grammar','method','seed'],key))|{'best_reward':b.reward,'operations':b.operations,'turns':b.turns,'evaluation_at_best':b.evaluation,'program':b.program,'ringdowns':b.n_ringdowns,'bursts':b.n_bursts})
    return pd.DataFrame(out)

def plot_convergence(df):
    fig,ax=plt.subplots(figsize=(10,5))
    for (grammar,method),g in df[df['map']=='reference'].groupby(['grammar','method']):
        curves=[]
        for seed,z in g.groupby('seed'):
            y=z.sort_values('evaluation').reward.ffill().fillna(-np.inf).to_numpy();curves.append(np.maximum.accumulate(y))
        x=np.arange(1,len(curves[0])+1);c=np.array(curves)
        ax.plot(x,c.mean(0),label=f'{grammar}/{method}')
        if len(c)>1:ax.fill_between(x,c.mean(0)-c.std(0),c.mean(0)+c.std(0),alpha=.15)
    ax.set(xlabel='Actual physical evaluations per run',ylabel='Best observed curl',title='Grammar optimization convergence');ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout();return fig

def plot_champions(df,folder):
    fig,axs=plt.subplots(2,2,figsize=(13,7),sharex=True)
    for ax,((grammar,method),g) in zip(axs.flat,df[(df['map']=='reference')&df.valid].groupby(['grammar','method'])):
        b=g.loc[g.reward.idxmax()];z=get_program(folder,b);t=np.linspace(0,1,900)
        ax.plot(t,fields(z,900)[0],lw=1.6)
        ax.set(title=f'{grammar} / {method}: R={b.reward:.4f}, ops={b.operations}',ylabel='E/Ec',xlabel='Normalized time');ax.grid(alpha=.2)
        ax.text(.02,.02,b.program,transform=ax.transAxes,fontsize=7,va='bottom',wrap=True)
    fig.tight_layout();return fig

def plot_pareto(df):
    fig,ax=plt.subplots(figsize=(9,5))
    for (grammar,method),g in df[(df['map']=='reference')&df.valid].groupby(['grammar','method']):
        best=-np.inf;xx=[];yy=[]
        for k in sorted(g.operations.unique()):
            v=g.loc[g.operations==k,'reward'].max()
            if v>best:best=v;xx.append(k);yy.append(v)
        ax.step(xx,yy,where='post',marker='o',label=f'{grammar}/{method}')
    ax.set(xlabel='Number of primitive operations',ylabel='Max curl',title='Measured reward–program complexity Pareto');ax.grid(alpha=.2);ax.legend(fontsize=8);fig.tight_layout();return fig

def plot_evolution(df,folder):
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for ax,((grammar,method),g) in zip(axes.flat,df[(df['map']=='reference')&(df.seed==0)].groupby(['grammar','method'])):
        g=g.sort_values('evaluation');inc=-np.inf;waves=[]
        for _,r in g.iterrows():
            if r.valid and np.isfinite(r.reward) and r.reward>inc:inc=r.reward;champ=get_program(folder,r)
            elif not waves:champ=get_program(folder,r)
            waves.append(fields(champ,256)[0]);
        im=ax.imshow(waves,origin='lower',aspect='auto',extent=[0,1,1,len(waves)],interpolation='nearest')
        ax.set(xlabel='Normalized time',ylabel='Evaluation',title=f'{grammar}/{method} incumbent E(t)');fig.colorbar(im,ax=ax,shrink=.75)
    return fig

def plot_disorder(df):
    summ=summary(df);fig,axes=plt.subplots(1,2,figsize=(12,4))
    for (grammar,method),g in summ.groupby(['grammar','method']):
        means=g.groupby('map').best_reward.mean().sort_index()
        axes[0].plot(range(len(means)),means.values,label=f'{grammar}/{method}',marker='.',lw=1)
        axes[1].scatter(g.operations,g.best_reward,label=f'{grammar}/{method}',alpha=.5,s=14)
    axes[0].set(title='Mean reward across disorder maps',xlabel='Disorder index',ylabel='Best reward')
    axes[1].set(title='Champion complexity across disorder',xlabel='Operations',ylabel='Best reward')
    for ax in axes:ax.grid(alpha=.2);ax.legend(fontsize=7)
    fig.tight_layout();return fig

def variance(df):
    summ=summary(df);rows=[]
    for (grammar,method),g in summ.groupby(['grammar','method']):
        means=g.groupby('map').best_reward.mean();within=g.groupby('map').best_reward.var(ddof=1)
        rows.append(dict(grammar=grammar,method=method,across_disorder=float(means.var(ddof=1)) if len(means)>1 else np.nan,within_disorder_across_seeds=float(within.mean()) if within.notna().any() else np.nan))
    return pd.DataFrame(rows)

def time_to_target(df,fraction=.95):
    rows=[]
    for mp,g in df.groupby('map'):
        valid=g[g.valid & np.isfinite(g.reward)];
        if not len(valid):continue
        baseline=valid[valid.generation==0].reward.min();goal=baseline+fraction*(valid.reward.max()-baseline)
        for (grammar,method,seed),q in g.groupby(['grammar','method','seed']):
            hits=q[q.valid & (q.reward>=goal)]
            rows.append(dict(map=mp,grammar=grammar,method=method,seed=seed,threshold=goal,fraction=fraction,first_evaluation=float(hits.evaluation.min()) if len(hits) else np.nan))
    return pd.DataFrame(rows)

def transfer(df,folder,maps,oracle_method='rk4',fidelity=1):
    rows=[];summ=summary(df)
    for _,r in summ.iterrows():
        q=df[(df['map']==r['map'])&(df.grammar==r.grammar)&(df.method==r.method)&(df.seed==r.seed)]
        best=q.loc[q.reward.idxmax()];z=get_program(folder,best);wave=fields(z,900)
        for target,d in maps.items():
            v=evaluate_fields(wave,defects_ec=d,method=oracle_method,fidelity=fidelity)
            rows.append(dict(source=r['map'],target=target,grammar=r.grammar,method=r.method,seed=int(r.seed),reward=float(v['reward'][0]),valid=bool(v['valid'][0] and not v['native_guard_exceeded'][0])))
    return pd.DataFrame(rows)

def plot_transfer(tr):
    fig,axes=plt.subplots(2,2,figsize=(12,10),layout='constrained')
    for ax,((grammar,method),g) in zip(axes.flat,tr.groupby(['grammar','method'])):
        p=g.pivot_table(index='source',columns='target',values='reward',aggfunc='mean');im=ax.imshow(p,aspect='auto');fig.colorbar(im,ax=ax,shrink=.7)
        ax.set(title=f'{grammar}/{method}',xlabel='Target disorder',ylabel='Source disorder')
    return fig

def motif_frequency(df):
    records=[]
    for _,r in df[df.valid].iterrows():
        ops=r.program.split(' > ')
        for op in ops:records.append((r['grammar'],r.method,op,r.reward))
    return pd.DataFrame(records,columns=['grammar','method','operation','reward']).groupby(['grammar','method','operation']).agg(count=('reward','size'),mean_reward=('reward','mean')).reset_index()

def refinement(df,folder,maps,method='rk4',fidelity=2):
    rows=[]
    for _,b in summary(df).iterrows():
        if b['map'] not in maps:continue
        q=df[(df['map']==b['map'])&(df.grammar==b.grammar)&(df.method==b.method)&(df.seed==b.seed)]
        z=get_program(folder,q.loc[q.reward.idxmax()]);v=evaluate_fields(fields(z,900),defects_ec=maps[b['map']],method=method,fidelity=fidelity)
        rows.append(dict(map=b['map'],grammar=b.grammar,method=b.method,seed=b.seed,original=b.best_reward,refined=float(v['reward'][0]),valid=bool(v['valid'][0])))
    return pd.DataFrame(rows)

def export(df,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);d=folder/'analysis';d.mkdir(exist_ok=True)
    summary(df).to_csv(d/'run_summary.csv',index=False)
    variance(df).to_csv(d/'variance.csv',index=False)
    time_to_target(df,.95).to_csv(d/'target_95.csv',index=False)
    time_to_target(df,.99).to_csv(d/'target_99.csv',index=False)
    motif_frequency(df).to_csv(d/'motif_frequencies.csv',index=False)
    for name,fn in [('convergence',lambda:plot_convergence(df)),('champions',lambda:plot_champions(df,folder)),('pareto',lambda:plot_pareto(df)),('evolution',lambda:plot_evolution(df,folder)),('disorder',lambda:plot_disorder(df))]:
        fig=fn();fig.savefig(d/(name+'.png'),dpi=150);plt.close(fig)
    return d


In [ ]:
%%writefile FerroSim_Grammar_v3/shared_figures.py
"""
FerroSim v3: shared analysis & figure protocol.
Embedded verbatim in the v3 notebooks for standalone Anaconda/Jupyter execution.
Methods differ among representations, but every figure has the same scientific meaning.
No synthetic rewards; every optimization curve uses saved oracle evaluations.
"""
from __future__ import annotations
from functools import lru_cache
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _valid(df):
    return df[df.valid & np.isfinite(df.reward)].copy()

def _front(df):
    z=_valid(df).sort_values(["complexity","reward"],ascending=[True,False])
    if z.empty:return z
    keep=[]; incumbent=-np.inf
    for complexity,g in z.groupby("complexity",sort=True):
        v=g.iloc[0]
        if float(v.reward)>incumbent+1e-12:
            keep.append(v);incumbent=float(v.reward)
    return pd.DataFrame(keep) if keep else z.iloc[:0]

def _incumbent(g):
    g=g.sort_values("evaluation")
    vals=np.where(g.valid & np.isfinite(g.reward),g.reward,-np.inf)
    return g, np.maximum.accumulate(vals)

def _save(fig,out,name):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    fig.savefig(out/name,dpi=150,bbox_inches="tight")
    plt.show();plt.close(fig)
    return out/name

def _champion(g):
    z=_valid(g)
    return None if z.empty else z.loc[z.reward.idxmax()]

def _scalar(row,key,default=np.nan):
    try:return float(row[key])
    except (TypeError,ValueError,KeyError):return default

def _zero_reward(ctx,map_id):
    cache=ctx.setdefault("zero_cache",{})
    if map_id not in cache:
        from oracle import evaluate_fields
        result=evaluate_fields(np.zeros(900),defects_ec=ctx["defect"](map_id),
                               method="rk4",max_substeps=ctx["max_substeps"])
        cache[map_id]=float(result["reward"][0]) if bool(result["valid"][0]) and not bool(result["native_guard_exceeded"][0]) else np.nan
    return cache[map_id]

def _selected(ctx,map_id):
    d=_valid(ctx["data"])
    z=d[d["map"]==map_id]
    if z.empty:return {}
    ans={"best":z.loc[z.reward.idxmax()]}
    simple=z[z.complexity<=1]
    if len(simple):ans["simple"]=simple.loc[simple.reward.idxmax()]
    return ans

def _grid(count,maxcols=5):
    cols=max(1,min(maxcols,int(count)))
    return int(math.ceil(count/cols)),cols

def figure01_disorders(ctx):
    maps=ctx["maps"];r,c=_grid(len(maps))
    fig,axs=plt.subplots(r,c,figsize=(3.1*c,2.9*r),squeeze=False,layout="constrained")
    for ax in axs.flat:ax.axis("off")
    for ax,mid in zip(axs.flat,maps):
        z=ctx["defect"](mid);ax.imshow(np.linalg.norm(z,axis=2),origin="lower")
        ax.set_title(mid);ax.axis("on")
    _save(fig,ctx["figdir"],"F01_disorder_maps.png")

def figure02_convergence(ctx):
    df=ctx["data"];maps=ctx["maps"];methods=ctx["methods"]
    fig,axs=plt.subplots(1,2 if len(maps)>1 else 1,figsize=(12 if len(maps)>1 else 7,4.4),squeeze=False)
    ax=axs[0,0]
    for meth in methods:
        g=df[(df["map"]==maps[0])&(df.method_key==meth)]
        for seed,s in g.groupby("seed"):
            s,b=_incumbent(s); ax.step(s.evaluation,np.where(np.isfinite(b),b,np.nan),
                                    where="post",alpha=.45,label=meth if seed==g.seed.min() else None)
    ax.set(xlabel="Attempted oracle evaluations / run",ylabel="Best measured reward",title="Reference-map convergence")
    ax.grid(alpha=.2);ax.legend(fontsize=7)
    if len(maps)>1:
        ax=axs[0,1]
        for meth in methods:
            traces=[]
            for (mid,seed),g in df[df.method_key==meth].groupby(["map","seed"]):
                g,b=_incumbent(g);traces.append(b)
            if not traces:continue
            minlen=min(map(len,traces));a=np.array([t[:minlen] for t in traces]);a[~np.isfinite(a)]=np.nan
            mu=np.nanmean(a,axis=0);n=np.sum(np.isfinite(a),axis=0)
            se=np.nanstd(a,axis=0,ddof=1)/np.sqrt(np.maximum(n,1)) if len(a)>1 else np.zeros(minlen)
            ax.plot(np.arange(1,minlen+1),mu,label=meth)
            if len(a)>1:ax.fill_between(np.arange(1,minlen+1),mu-se,mu+se,alpha=.12)
        ax.set(xlabel="Attempted oracle evaluations / run",ylabel="Mean best reward",title="Disorder + seed ensemble (descriptive)")
        ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,ctx["figdir"],"F02_reward_vs_evaluations.png")

def figure03_reference_pareto(ctx):
    df=ctx["data"];g=df[df["map"]==ctx["maps"][0]]
    fig,ax=plt.subplots(figsize=(8,5))
    for meth,s in g.groupby("method_key"):
        v=_valid(s);ax.scatter(v.complexity,v.reward,s=8,alpha=.15)
        f=_front(s)
        if len(f):ax.plot(f.complexity,f.reward,"o-",lw=1.5,ms=3,label=meth)
    ax.set(xlabel=ctx["complexity_label"],ylabel="Measured terminal curl",
           title="F03 — Reference-disorder reward–complexity Pareto fronts")
    ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,ctx["figdir"],"F03_reference_pareto.png")

def figure04_waveform_histories(ctx,history_steps=40):
    """One waveform-history figure per disorder, with one panel per algorithm."""
    df=ctx["data"]
    for mid in ctx["maps"]:
        methods=ctx["methods"];rows,cols=_grid(len(methods),maxcols=2)
        fig,axs=plt.subplots(rows,cols,figsize=(6*cols,3.4*rows),squeeze=False,layout="constrained")
        for ax in axs.flat:ax.axis("off")
        for ax,meth in zip(axs.flat,methods):
            g=_valid(df[(df["map"]==mid)&(df.method_key==meth)&(df.seed==df.seed.min())])
            if g.empty:continue
            g=g.sort_values("evaluation")
            checkpoints=np.unique(np.linspace(0,len(g)-1,min(history_steps,len(g)),dtype=int))
            best=g.reward.to_numpy()
            idx=[]; incumbent=-np.inf;winner=0
            for i,y in enumerate(best):
                if y>incumbent:incumbent=y;winner=i
                idx.append(winner)
            rows_to_plot=[g.iloc[idx[i]] for i in checkpoints]
            waves=np.stack([ctx["field"](row) for row in rows_to_plot])
            amp=max(float(np.max(np.abs(waves))),1e-8)
            ax.imshow(waves,aspect="auto",origin="lower",cmap="coolwarm",vmin=-amp,vmax=amp,
                      extent=[0,1,float(g.evaluation.iloc[checkpoints[0]]),float(g.evaluation.iloc[checkpoints[-1]])])
            ax.axis("on");ax.set(title=meth,xlabel="Normalized control time",ylabel="Evaluation of incumbent")
        fig.suptitle("F04 — Best-so-far waveform history: "+str(mid))
        _save(fig,ctx["figdir"],"F04_waveform_history_"+str(mid)+".png")

def figure05_winners(ctx):
    picks=_selected(ctx,ctx["maps"][0]);fig,axs=plt.subplots(1,2,figsize=(12,4.3))
    for label,row in picks.items():
        w=ctx["field"](row)
        axs[0].plot(np.linspace(0,1,len(w)),w,label=f"{label}, R={row.reward:.4g}, C={int(row.complexity)}")
        axs[1].plot(np.linspace(0,1,len(w)),w/max(np.max(np.abs(w)),1e-12),label=label)
    axs[0].set(xlabel="Normalized control time",ylabel="E/Ec",title="Physical amplitude")
    axs[1].set(xlabel="Normalized control time",ylabel="Normalized E/Ec",title="Shape only (not reward comparable)")
    for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=8)
    _save(fig,ctx["figdir"],"F05_best_and_simple_waveforms.png")

def figure06_incumbent_diagnostics(ctx):
    df=ctx["data"]; fig,axs=plt.subplots(2,2,figsize=(12,8),layout="constrained")
    labels=[("reward","Best reward"),("complexity",ctx["complexity_label"]),
            ("peak_ec","Peak |E| / Ec"),("rms_ec","RMS E / Ec")]
    for meth,g in df[(df["map"]==ctx["maps"][0])&(df.seed==df.seed.min())].groupby("method_key"):
        g=g.sort_values("evaluation");inc=-np.inf;winner=None;traces={k:[] for k,_ in labels}
        for _,row in g.iterrows():
            if bool(row.valid) and np.isfinite(row.reward) and row.reward>inc:
                inc=float(row.reward);winner=row
            for key,_ in labels:traces[key].append(_scalar(winner,key) if winner is not None else np.nan)
        for ax,(key,_) in zip(axs.flat,labels):
            ax.plot(g.evaluation,traces[key],label=meth)
    for ax,(key,title) in zip(axs.flat,labels):
        ax.set(xlabel="Attempted evaluations / run",ylabel=title,title=title);ax.grid(alpha=.2);ax.legend(fontsize=6)
    _save(fig,ctx["figdir"],"F06_incumbent_field_complexity_reward.png")

def figure07_surrogate_calibration(ctx):
    d=_valid(ctx["data"]);d=d[(d.generation>0)&np.isfinite(d.predicted_mean)&np.isfinite(d.predicted_std)]
    if d.empty:
        print("F07 skipped: no finite held-out proposal predictions.");return
    fig,axs=plt.subplots(1,2,figsize=(11,4.3))
    for meth,g in d.groupby("method_key"):
        sample=g.iloc[::max(1,len(g)//250)]
        axs[0].scatter(sample.predicted_mean,sample.reward,s=8,alpha=.3,label=meth)
    lo=min(float(d.predicted_mean.min()),float(d.reward.min()));hi=max(float(d.predicted_mean.max()),float(d.reward.max()))
    axs[0].plot([lo,hi],[lo,hi],"k--",lw=1)
    axs[0].set(xlabel="DKL predicted reward before measurement",ylabel="Actual oracle reward",title="Proposal calibration (selection-biased)")
    for meth,g in d.groupby("method_key"):
        e=np.abs(g.reward-g.predicted_mean).to_numpy()
        std=np.maximum(g.predicted_std.to_numpy(),1e-12)
        axs[1].plot([1,2,3],[np.mean(e<=k*std) for k in (1,2,3)],"o-",label=meth)
    from math import erf,sqrt
    axs[1].plot([1,2,3],[erf(k/sqrt(2)) for k in (1,2,3)],"k--",label="Gaussian nominal")
    axs[1].set(xlabel="Credible interval ± k × std",ylabel="Empirical fraction covered",ylim=(0,1.03),
               title="On-policy selected candidates; not IID holdout")
    for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,ctx["figdir"],"F07_DKL_prediction_calibration.png")

def figure08_09_10_physical(ctx):
    if not ctx.get("run_physical",True):
        print("F08–F10 physical revalidation disabled by configuration.");return
    from oracle import evaluate_fields,curl_field
    picks=_selected(ctx,ctx["maps"][0])
    if not picks:return
    row=picks["best"];wave=ctx["field"](row);defects=ctx["defect"](ctx["maps"][0])
    v=evaluate_fields(wave,defects_ec=defects,method="rk4",
                      return_history=True,max_substeps=ctx["max_substeps"])
    if not bool(v["valid"][0]) or bool(v["native_guard_exceeded"][0]):
        print("F08–F10 skipped: champion invalid under independent reevaluation.");return
    P=v["final_P"][0];curl=np.abs(curl_field(P))
    fig,axs=plt.subplots(1,3,figsize=(13,3.8),layout="constrained")
    for ax,arr,title in zip(axs,[P[0],P[1],curl],["P_x","P_y","|curl P|"]):
        im=ax.imshow(arr,origin="lower");ax.set_title(title);fig.colorbar(im,ax=ax,shrink=.75)
    _save(fig,ctx["figdir"],"F08_terminal_polarization_fields.png")
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    axs[0].plot(v["time"],v["curl_history"][0],label="whole grid")
    axs[0].plot(v["time"],v["interior_history"][0],label="interior only")
    axs[0].set(xlabel="Physical time",ylabel="Total |curl|",title="Reward formation")
    axs[1].plot(v["time"],v["mean_P"][0,0],label="Mean P_x")
    axs[1].plot(v["time"],v["mean_P"][0,1],label="Mean P_y")
    axs[1].set(xlabel="Physical time",ylabel="Mean polarization",title="Polarization dynamics")
    for ax in axs:ax.grid(alpha=.2);ax.legend()
    _save(fig,ctx["figdir"],"F09_curl_and_polarization_trajectories.png")
    checks=[("fidelity 1",float(v["reward"][0]),float(v["interior"][0]),True)]
    if ctx.get("run_refinement",True):
        z=evaluate_fields(wave,defects_ec=defects,method="rk4",fidelity=2,
                          max_substeps=max(10000,2*ctx["max_substeps"]))
        good=bool(z["valid"][0]) and not bool(z["native_guard_exceeded"][0])
        checks.append(("fidelity 2",float(z["reward"][0]) if good else np.nan,
                      float(z["interior"][0]) if good else np.nan,good))
    pd.DataFrame([dict(fidelity=f,whole=r,interior=i,valid=valid,boundary=r-i)
                  for f,r,i,valid in checks]).to_csv(Path(ctx["figdir"])/"F10_numerical_validation.csv",index=False)
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    pos=np.arange(len(checks)); whole=[z[1] for z in checks];inside=[z[2] for z in checks]
    axs[0].bar(pos,inside,label="Interior")
    axs[0].bar(pos,np.array(whole)-inside,bottom=inside,label="Boundary")
    axs[0].set(xticks=pos,xticklabels=[z[0] for z in checks],ylabel="Total absolute curl",
               title="Integration fidelity / boundary decomposition");axs[0].legend()
    fine=np.linspace(0,1,1801);coarse=np.linspace(0,1,len(wave))
    reference=ctx["field"](row,1801);interp=np.interp(fine,coarse,wave)
    axs[1].plot(fine,reference-interp,lw=1);axs[1].set(xlabel="Normalized control time",
                     ylabel="Fine rendering − 900-sample interpolation",
                     title="Waveform resolution (diagnostic only)")
    for ax in axs:ax.grid(alpha=.2)
    _save(fig,ctx["figdir"],"F10_numerical_and_sampling_diagnostics.png")

def figure11_18_ensemble(ctx):
    maps=ctx["maps"]
    if len(maps)<=1:
        print("F11–F18 skipped: N=1 cannot estimate disorder variability or cross-disorder generalization.")
        return
    df=ctx["data"];valid=_valid(df);methods=ctx["methods"];out=ctx["figdir"]
    runbest=valid.sort_values("reward").groupby(["map","method_key","seed"],sort=False).tail(1).copy()
    # F11: one observation per disorder per method, averaging optimizer seeds.
    means=runbest.groupby(["map","method_key"],as_index=False).reward.mean()
    fig,ax=plt.subplots(figsize=(10,4.5))
    vals=[means[means.method_key==m].reward.to_numpy() for m in methods]
    ax.boxplot(vals,showmeans=True);ax.set_xticks(np.arange(1,len(methods)+1),methods,rotation=30)
    ax.set(ylabel="Best measured reward",title="F11 — Per-disorder best reward (averaged over seeds)")
    ax.grid(alpha=.2);_save(fig,out,"F11_reward_distributions.png")
    # F12 paired difference on the SAME disorder maps, against GA where present.
    baseline=next((m for m in methods if "GA" in m.upper() and "QD" not in m.upper()),methods[0])
    pivot=means.pivot(index="map",columns="method_key",values="reward").reindex(index=maps)
    fig,ax=plt.subplots(figsize=(9,4.7))
    for m in methods:
        if m==baseline:continue
        d=pivot[m]-pivot[baseline]
        ax.plot(range(len(maps)),d,"o-",label=m)
    ax.axhline(0,color="black",lw=1);ax.set(xticks=range(len(maps)),xticklabels=maps,
        ylabel="Paired reward difference",title="F12 — Method minus "+baseline)
    ax.tick_params(axis="x",rotation=70);ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,out,"F12_paired_reward_differences.png")
    # F13 winning complexity across disorders.
    fig,ax=plt.subplots(figsize=(10,4.5))
    for m in methods:
        sub=runbest[runbest.method_key==m]
        x=[maps.index(v) for v in sub["map"]];ax.scatter(x,sub.complexity,s=22,alpha=.5,label=m)
    ax.set(xticks=range(len(maps)),xticklabels=maps,ylabel=ctx["complexity_label"],
           title="F13 — Winner complexity versus disorder")
    ax.tick_params(axis="x",rotation=70);ax.grid(alpha=.2);ax.legend(fontsize=7)
    _save(fig,out,"F13_winner_complexity.png")
    # F14 fraction of observed gain retained with <=1 complexity unit.
    rows=[]
    for mid in maps:
        d=valid[valid["map"]==mid]
        if d.empty:continue
        best=float(d.reward.max());small=d[d.complexity<=1]
        zero=_zero_reward(ctx,mid)
        rows.append((mid,(float(small.reward.max())-zero)/(best-zero) if len(small) and np.isfinite(zero) and best>zero else np.nan))
    fig,ax=plt.subplots(figsize=(9,4.3))
    ax.bar([p[0] for p in rows],[p[1] for p in rows])
    ax.set(ylabel="Fraction of discovered improvement retained",
           title="F14 — Parsimony (complexity ≤1) at same disorder")
    ax.tick_params(axis="x",rotation=70);ax.grid(alpha=.2)
    _save(fig,out,"F14_simple_gain_retention.png")
    # F15: optional measured champion cross-disorder transfer, never surrogate predictions.
    if ctx.get("run_transfer",False):
        from oracle import evaluate_fields
        transfers=[]
        for source in maps:
            chosen=_selected(ctx,source)
            if not chosen:continue
            for kind,row in chosen.items():
                field=ctx["field"](row)
                for target in maps:
                    physical=evaluate_fields(field,defects_ec=ctx["defect"](target),
                                             max_substeps=ctx["max_substeps"])
                    good=bool(physical["valid"][0]) and not bool(physical["native_guard_exceeded"][0])
                    transfers.append(dict(source=source,target=target,kind=kind,
                        reward=float(physical["reward"][0]) if good else np.nan,valid=good))
        tr=pd.DataFrame(transfers);tr.to_csv(Path(out)/"F15_transfer_records.csv",index=False)
        fig,axs=plt.subplots(1,len(tr.kind.unique()),figsize=(5.5*len(tr.kind.unique()),5),squeeze=False)
        for ax,kind in zip(axs.flat,tr.kind.unique()):
            pivot=tr[(tr.kind==kind)&tr.valid].pivot(index="source",columns="target",values="reward").reindex(index=maps,columns=maps)
            im=ax.imshow(pivot,aspect="auto")
            ax.set(title=kind,xticks=range(len(maps)),xticklabels=maps,yticks=range(len(maps)),yticklabels=maps)
            ax.tick_params(axis="x",rotation=90);fig.colorbar(im,ax=ax)
        fig.suptitle("F15 — Validated measured cross-disorder transfer")
        _save(fig,out,"F15_cross_disorder_transfer.png")
    else:print("F15 transfer skipped; enable RUN_TRANSFER to perform expensive extra oracle evaluations.")
    # F16 random-effects-inspired corrected between-disorder variance.
    stats=[]
    for meth,g in runbest.groupby("method_key"):
        means_by_map=g.groupby("map").reward.mean()
        within=g.groupby("map").reward.var(ddof=1)
        nseeds=g.groupby("map").reward.size()
        variance_mean=float(means_by_map.var(ddof=1))
        sampling=float((within/nseeds).mean()) if within.notna().any() else np.nan
        stats.append(dict(method=meth,observed_between_map_mean_variance=variance_mean,
                     mean_within_seed_variance=float(within.mean()) if within.notna().any() else np.nan,
                     corrected_between_estimate=variance_mean-sampling if np.isfinite(sampling) else np.nan))
    stats=pd.DataFrame(stats);stats.to_csv(Path(out)/"F16_variance_decomposition.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,4.5))
    xx=np.arange(len(stats));ax.bar(xx-.18,stats.observed_between_map_mean_variance,width=.36,label="Between map means")
    ax.bar(xx+.18,stats.mean_within_seed_variance,width=.36,label="Within-map optimizer seeds")
    ax.set(xticks=xx,xticklabels=stats.method,ylabel="Variance of reward",title="F16 — Descriptive variance components")
    ax.tick_params(axis="x",rotation=30);ax.legend(fontsize=8);ax.grid(alpha=.2)
    _save(fig,out,"F16_variance_components.png")
    # F17 fixed pooled target per map; unsuccessful runs treated as censored.
    records=[]
    for mid in maps:
        g=valid[valid["map"]==mid]
        if g.empty:continue
        zero=_zero_reward(ctx,mid);top=float(g.reward.max())
        threshold=zero+.95*(top-zero)
        for (meth,seed),full in df[df["map"]==mid].groupby(["method_key","seed"]):
            hit=full[full.valid & (full.reward>=threshold)]
            records.append(dict(map=mid,method=meth,seed=seed,reached=not hit.empty,
                                first_evaluation=int(hit.evaluation.min()) if not hit.empty else np.nan))
    events=pd.DataFrame(records);events.to_csv(Path(out)/"F17_target_attainment.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,4.5))
    horizon=int(df.evaluation.max())
    for meth,g in events.groupby("method"):
        xs=np.arange(1,horizon+1)
        times=g.first_evaluation.dropna().to_numpy()
        yy=np.array([(times<=i).sum()/len(g) for i in xs])
        ax.step(xs,yy,where="post",label=meth)
    ax.set(xlabel="Attempted oracle evaluations",ylabel="Fraction of runs reaching target",
           ylim=(0,1.05),title="F17 — 95% pooled target attainment (unreached = censored)")
    ax.grid(alpha=.2);ax.legend(fontsize=8);_save(fig,out,"F17_time_to_target.png")
    # F18: common physical disorder descriptor and achieved reward.
    records=[]
    for mid in maps:
        a=ctx["defect"](mid);pos=np.argwhere(np.linalg.norm(a,axis=2)>0)
        if len(pos)<2:continue
        dist=np.sqrt(np.sum((pos[:,None,:]-pos[None,:,:])**2,axis=2))
        np.fill_diagonal(dist,np.inf)
        records.append(dict(map=mid,mean_neighbor_distance=float(np.mean(dist.min(axis=1))),
                            max_reward=float(valid[valid["map"]==mid].reward.max())))
    desc=pd.DataFrame(records);desc.to_csv(Path(out)/"F18_disorder_descriptors.csv",index=False)
    fig,ax=plt.subplots(figsize=(8,4.5))
    for _,r in desc.iterrows():
        ax.scatter(r.mean_neighbor_distance,r.max_reward)
        ax.annotate(r["map"],(r.mean_neighbor_distance,r.max_reward),fontsize=7)
    ax.set(xlabel="Mean nearest defect-site distance (pixels)",ylabel="Maximum observed curl",
           title="F18 — Disorder geometry and accessible reward")
    ax.grid(alpha=.2);_save(fig,out,"F18_disorder_structure_vs_reward.png")

def run_common(ctx):
    d=ctx["data"]
    if len(d)==0:raise RuntimeError("No measured oracle histories; execute the experiment cell first.")
    for k in ("map","method_key","seed","evaluation","generation","reward","valid","complexity","peak_ec","rms_ec","predicted_mean","predicted_std"):
        if k not in d.columns:raise KeyError("Missing common data column "+k)
    figure01_disorders(ctx)
    figure02_convergence(ctx)
    figure03_reference_pareto(ctx)
    figure04_waveform_histories(ctx)
    figure05_winners(ctx)
    figure06_incumbent_diagnostics(ctx)
    figure07_surrogate_calibration(ctx)
    figure08_09_10_physical(ctx)
    figure11_18_ensemble(ctx)
    print("Standard v3 figures saved:",Path(ctx["figdir"]).resolve())

def appendable_per_disorder_figures(ctx,only_disorders=None,history_steps=30):
    """
    F19: per-disorder best-so-far waveform history for each algorithm.
    F20: per-disorder reward–complexity Pareto fronts with method and pooled fronts.
    Can be called as the last code cell in any FerroSim v3 notebook.
    """
    df=ctx["data"];out=Path(ctx["figdir"]);maps=ctx["maps"] if only_disorders is None else list(only_disorders)
    for mid in maps:
        if mid not in ctx["maps"]:raise ValueError(f"Unknown disorder {mid}")
        current=df[df["map"]==mid]
        if current.empty:continue
        methods=ctx["methods"];fig,axs=plt.subplots(len(methods),2,figsize=(12,3.1*len(methods)),squeeze=False,layout="constrained")
        for i,meth in enumerate(methods):
            g=current[(current.method_key==meth)&(current.seed==current.seed.min())].sort_values("evaluation")
            if g.empty:continue
            valid=_valid(g)
            if valid.empty:continue
            checkpoint=np.unique(np.linspace(0,len(g)-1,min(history_steps,len(g)),dtype=int))
            bestrow=None;best=-np.inf;wave=[]
            checkpoints=set(checkpoint.tolist())
            for j,(_,r) in enumerate(g.iterrows()):
                if bool(r.valid) and np.isfinite(r.reward) and float(r.reward)>best:
                    best=float(r.reward);bestrow=r
                if j in checkpoints:wave.append(ctx["field"](bestrow) if bestrow is not None else np.zeros(900))
            waves=np.asarray(wave);extent=[0,1,int(g.evaluation.iloc[checkpoint[0]]),int(g.evaluation.iloc[checkpoint[-1]])]
            axs[i,0].imshow(waves,origin="lower",aspect="auto",extent=extent,cmap="coolwarm")
            axs[i,0].set(title=meth+" — incumbent waveform",xlabel="Control time/T",ylabel="Evaluation")
            p=_front(g);axs[i,1].scatter(valid.complexity,valid.reward,s=7,alpha=.18)
            if len(p):axs[i,1].plot(p.complexity,p.reward,"o-",ms=3)
            axs[i,1].set(title=meth+" — measured Pareto front",xlabel=ctx["complexity_label"],ylabel="Curl reward")
            axs[i,1].grid(alpha=.2)
        fig.suptitle("F19 — Per-disorder waveform and method Pareto history: "+str(mid))
        _save(fig,out,"F19_history_and_method_pareto_"+str(mid)+".png")
    rows,cols=_grid(len(maps))
    fig,axs=plt.subplots(rows,cols,figsize=(4.3*cols,3.4*rows),squeeze=False,layout="constrained")
    for ax in axs.flat:ax.axis("off")
    for ax,mid in zip(axs.flat,maps):
        part=df[df["map"]==mid]
        for meth,g in part.groupby("method_key"):
            v=_valid(g);ax.scatter(v.complexity,v.reward,s=4,alpha=.08)
            f=_front(g)
            if len(f):ax.plot(f.complexity,f.reward,lw=1,marker=".",label=meth)
        globalfront=_front(part)
        if len(globalfront):ax.plot(globalfront.complexity,globalfront.reward,"k--",lw=2,label="pooled")
        ax.axis("on");ax.set(title=mid,xlabel=ctx["complexity_label"],ylabel="Curl reward");ax.grid(alpha=.2)
    if len(maps)<=4:axs.flat[0].legend(fontsize=6)
    fig.suptitle("F20 — Pareto fronts across disorder realizations (within-representation)")
    _save(fig,out,"F20_all_disorder_pareto_fronts.png")


## Reference defect lattice, dynamic maps and isolated source fingerprints

In [ ]:
REF_B85="c-oun7wQ`j$;eQ~P_3SlTAW;@Zl$1ZlV+i=qoAIaUsO_*m=~X4l#&V(cT3DEP6dh=XCxM+0{I#y1_n9`Mw&VbwF+bcE{0JsARsI7O?HNjtHWS%PcGN8!ix894#OpUE6&gKjJ-L0{6i|p`+h9lcB8k$XilRI$jDo~(<az&#D=fq!$%=rXIw_p*DwWLGy3~pKKB|n@%uK&eo@W_kJ0oxLcv9uMND&#51#gb(VCv40=%@VpVc*8TsMolj@b0hXZTd}(D~t$-+ia*3B8W-av06YqktT+b~F3hU00n))6XykMtK`gZQA5BY~uGoilo|;)}Yb!Its`FS69yYQe7TMv-I<ocLmp>uF>(8p#pejU*CT$cC`I83Qz$6MT1~F"
reference_bytes=zlib.decompress(base64.b85decode(REF_B85.encode()))
(ROOT/"defect_list_seed.npy").write_bytes(reference_bytes)
SOURCES={name:hashlib.sha256((ROOT/name).read_bytes()).hexdigest() for name in ("representation.py","oracle.py","dkl.py","experiment.py","analysis.py","shared_figures.py")}
for name in ("representation","oracle","dkl","experiment","analysis","shared_figures"):sys.modules.pop(name,None)
sys.path.insert(0,str(ROOT.resolve()))
from representation import fields,complexity
from oracle import evaluate_fields,default_defects
from shared_figures import run_common,appendable_per_disorder_figures
REF=default_defects();assert REF.shape==(20,20,2)
assert int(np.any(REF!=0,axis=2).sum())==19
MAP_DIR=ROOT/"disorder_maps";MAP_DIR.mkdir(exist_ok=True)
MAPS=["reference"]+[f"d{k:02d}" for k in range(N_DISORDERS-1)]
np.save(MAP_DIR/"reference.npy",REF)
active=REF.reshape(-1,2)[np.any(REF.reshape(-1,2)!=0,axis=1)]
for j,mid in enumerate(MAPS[1:]):
    rg=np.random.default_rng(20261122+j);loc=rg.choice(400,19,replace=False)
    v=np.zeros((400,2));v[loc]=active[rg.permutation(19)]
    np.save(MAP_DIR/(mid+".npy"),v.reshape(20,20,2))
def defect_for(mid):return np.load(MAP_DIR/(mid+".npy"),allow_pickle=False)
print("Maps:",len(MAPS),"Sources:",SOURCES)

In [ ]:
from experiment import METHODS,GRAMMARS
config=dict(representation="Grammar",mode=MODE,N_DISORDERS=N_DISORDERS,N_SEEDS=N_SEEDS,generations=GENERATIONS,initial=INITIAL_CONTROLS,batch=BATCH_SIZE,candidates=CANDIDATES,source=SOURCES,defects=hashlib.sha256(reference_bytes).hexdigest())
RID=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
RESULTS=ROOT/"experiments"/RID;RESULTS.mkdir(parents=True,exist_ok=True)
(RESULTS/"config.json").write_text(json.dumps(config,indent=2))
cmd=[sys.executable,str(ROOT/"experiment.py"),"--out",str(RESULTS),
     "--maps",str(N_DISORDERS),"--seeds",str(N_SEEDS),
     "--generations",str(GENERATIONS),"--initial",str(INITIAL_CONTROLS),
     "--batch",str(BATCH_SIZE),"--candidates",str(CANDIDATES),
     "--steps","5" if MODE=="QUICK" else "35"]
cmd+=["--oracle-method","rk4"]
env=os.environ.copy();env.update({"OPENBLAS_NUM_THREADS":"1","NUMBA_NUM_THREADS":"1","OMP_NUM_THREADS":"1","MKL_NUM_THREADS":"1"})
subprocess.run(cmd,cwd=str(ROOT),env=env,check=True)
df=pd.read_csv(RESULTS/"all_evaluations.csv")
df["method_key"]=df["grammar"]+"/"+df["method"]
df["complexity"]=df["turns"].astype(int)
expected=N_DISORDERS*N_SEEDS*len(METHODS)*len(GRAMMARS)*(INITIAL_CONTROLS+BATCH_SIZE*GENERATIONS)
assert len(df)==expected,(len(df),expected)
assert set(df["map"])==set(MAPS)
print("Measured physical evaluations:",len(df),"valid:",int(df.valid.sum()))

## Load and normalize common measured histories

The same 20 figure names apply to all representations. The Pareto-complexity coordinate is **resolved waveform turning points** (2% prominence). The original grammar operation count and segment count remain in the measured CSV.

In [ ]:
from analysis import get_program
# Use the SAME disorder files as the actual optimization subprocess, not a regenerated estimate.
MAP_DIR=RESULTS/"disorder_maps"
MAPS=["reference"]+[f"d{k:02d}" for k in range(N_DISORDERS-1)]
def defect_for(mid):return np.load(MAP_DIR/(mid+".npy"),allow_pickle=False)
def physical_field(row):
    x=get_program(RESULTS,row)
    return np.asarray(fields(x,n=900),float).reshape(-1)
FIG_DIR=RESULTS/"figures";FIG_DIR.mkdir(exist_ok=True)
CTX=dict(data=df,maps=MAPS,methods=list(dict.fromkeys(df.method_key.tolist())),defect=defect_for,field=physical_field,
         figdir=FIG_DIR,max_substeps=100000,run_refinement=RUN_REFINEMENT,run_transfer=RUN_TRANSFER,
         run_physical=RUN_PHYSICAL_DIAGNOSTICS,
         complexity_label="Resolved waveform turns (2% prominence)")
run_common(CTX)
print("Figures:",FIG_DIR)

## Final appendable visualizations — F19 and F20

F19 plots the best-so-far **simulator-evaluated physical waveform** over the evaluation history for every algorithm and every selected disorder. In the adjacent panel, it plots the Pareto front of measured reward against resolved waveform complexity. F20 compares per-method and pooled empirical fronts across all disorder realizations. For N=1, the figure correctly contains one map. Choose `ONLY_DISORDERS = ['reference','d00']` to restrict. These are empirical sampled Pareto fronts, not certified global fronts.

In [ ]:
ONLY_DISORDERS=None        # None = all maps, or list of specific map IDs
HISTORY_SNAPSHOTS=30
if PLOT_ALL_DISORDERS:
    appendable_per_disorder_figures(CTX,only_disorders=ONLY_DISORDERS,history_steps=HISTORY_SNAPSHOTS)
print("Completed Grammar v3; no original notebooks overwritten.")